<a href="https://colab.research.google.com/github/talhanoor23/my_first_repository/blob/main/Copy_of_Copy_of_News_Agentic_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install langgraph langchain langsmith python-dotenv langchain-groq langchain_tavily langchain-community langchain-core
# !pip install --upgrade google-generativeai langchain-google-genai langchain

In [ ]:
import json

file_name = "/content/drive/MyDrive/Learnings/Agentic_AI/Code/Copy of Copy of News_Agentic_Bot.ipynb"

with open(file_name, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove broken widgets metadata
if "widgets" in nb.get("metadata", {}):
    nb["metadata"]["widgets"] = {}

with open(file_name, "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("Fixed notebook metadata!")

Fixed notebook metadata!


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash")

In [ ]:
from typing import Annotated, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

# class State(TypedDict):
#     messages: Annotated[list, add_messages]

class State(TypedDict):
    messages: list
    user_query: str
    # structured_memory: dict
    episodic_context: str
    final_answer: str

In [ ]:
os.environ["NEWS_API_KEY"]="70e32b43f4124e2895941e1bb7a109a2"
apiKey=os.environ["NEWS_API_KEY"]

In [ ]:
!pip install keybert
from keybert import KeyBERT

kw_model = KeyBERT(model="all-MiniLM-L6-v2")

def extract_keywords(text: str, top_k: int = 5) -> str:
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_k
    )
    return " OR ".join([kw[0] for kw in keywords])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import os
import requests
from typing import Optional, List, Dict
from datetime import datetime, timedelta, timezone
from langchain.tools import tool
from langchain_core.messages import BaseMessage


NEWS_API_URL = "https://newsapi.org/v2/everything"

In [ ]:
def fetch_financial_news(minutes: int, query: str) -> List[Dict]:
    from_time = (datetime.now(timezone.utc) - timedelta(days=7)).isoformat()

    params = {
        "language": "en",
        "pageSize": 10,
        "from": from_time,
        "q": query,
        "apiKey": os.getenv("NEWS_API_KEY"),
    }

    response = requests.get(NEWS_API_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json().get("articles", [])


In [ ]:
@tool
def financial_news_research(
    minutes: int = 1440,
    user_query: str = "",
    state: Optional[dict] = None
) -> dict:
    """
    Fetch financial news and mark state when done.
    """
    print("🚨 TOOL CALLED")
    if not user_query.strip():
        user_query = "finance market economy stocks"

    keyword_query = extract_keywords(user_query)
    articles = fetch_financial_news(minutes, keyword_query)

    if not articles:
        return {
            "results": [],
            "status": "NO_DATA",
            "message": "No relevant news found"
        }

    # LLM call happens once
    prompt_blocks = []
    valid_articles = []
    for i, a in enumerate(articles, start=1):
        title = a.get("title")
        if not title:
            continue
        valid_articles.append(a)
        description = a.get("description") or ""
        prompt_blocks.append(f"Article {i}\nTitle: {title}\nDescription: {description}\n")

    prompt = f"""
You are a financial market analyst.

For EACH article below, explain in one sentence
why it matters to financial markets.

Return a numbered list matching the article order.

{chr(10).join(prompt_blocks)}
"""
    response = llm.invoke(prompt).content.strip()
    explanations = [
        line.split(".", 1)[-1].strip()
        for line in response.splitlines()
        if line.strip()
    ]

    results = [
        {
            "headline": a.get("title"),
            "why_important": explanations[i] if i < len(explanations) else "No explanation",
            "source": a.get("source", {}).get("name"),
            "published_at": a.get("publishedAt"),
            "url": a.get("url"),
        }
        for i, a in enumerate(valid_articles)
    ]
    return results

In [ ]:
REASONING_SCHEMA = {
    "summary": str,
    "sentiment": "bullish | bearish | neutral",
    "confidence": float,  # 0.0 - 1.0
    "key_points": list[str],
    "risks": list[str],
    "time_horizon": "short_term | mid_term | long_term"
}

In [ ]:
REASONING_PROMPT = """
You are a financial reasoning agent.

Analyze the user's query and available context.

Return STRICT JSON matching this schema:
{
  "summary": string,
  "sentiment": "bullish" | "bearish" | "neutral",
  "confidence": number (0 to 1),
  "key_points": [string],
  "risks": [string],
  "time_horizon": "short_term" | "mid_term" | "long_term"
}

Rules:
- NO markdown
- NO explanations
- NO extra text
- If uncertain, lower confidence
"""


In [ ]:
import json
from langchain.tools import tool
from langchain_core.messages import HumanMessage

@tool
def structured_reasoning_tool(user_query: str) -> dict:
    """
    Perform structured financial reasoning on a user query.

    Input:
    - user_query: The user's financial or market-related question.

    Output (JSON):
    {
        "summary": string,
        "sentiment": "bullish" | "bearish" | "neutral",
        "confidence": number (0 to 1),
        "key_points": [string],
        "risks": [string],
        "time_horizon": "short_term" | "mid_term" | "long_term"
    }
    """
    print("🤖 REASONING AGENT CALLED")
    combined_prompt = f"{REASONING_PROMPT}\n\nUser Query: {user_query}"
    response = llm.invoke([
        HumanMessage(content=combined_prompt)
    ])
    print(response.content)

    try:
        return json.loads(response.content)
    except Exception:
        return {
            "summary": "Unable to generate structured reasoning",
            "sentiment": "neutral",
            "confidence": 0.0,
            "key_points": [],
            "risks": ["Parsing failure"],
            "time_horizon": "short_term"
        }

In [ ]:
SUPERVISOR_PROMPT = """
You are a financial AI system composed of specialized agents.

Your goal is to provide accurate, well-reasoned, and up-to-date responses.

You have access to:
- A Financial News Agent (real-time information)
- A Reasoning Agent (macro, cause-effect, implications, synthesis)
- Educational knowledge (no tools required)
If reasoning is needed, donot reason it yourself, first call the news the you must have to call the Reasoning Agent

General Principles:
- Do NOT invent facts, news, or events
- If real-time or external data is required, fetch it before reasoning
- If reasoning is required, base it strictly on known facts or fetched data
- Prefer multi-step reasoning over shallow answers
- You may use more than one agent if the user query requires it
- Agents may consume outputs from other agents

Tool & Agent Usage Rules:
- Use tools ONLY when real-time or external information is needed or is required
- Each tool may be called at most ONCE per query
- After a tool returns data, treat it as sufficient
- Do NOT retry or re-call same tools again for the same query
- If information remains incomplete, state uncertainty clearly

Decision Logic:
1. Determine if the query requires real-time data
   - If yes → call Financial News Agent
2. Determine if the query requires implications, interpretation, or synthesis
   - If yes → call Financial News Agent for News then call Reasoning Agent (using available data)
3. If neither is required → answer directly

Output:
- Provide a coherent final answer to the user
- Do not expose internal routing or agent decisions
- Do not mention tools unless explicitly asked
"""

In [ ]:
tools=[
        financial_news_research,
        structured_reasoning_tool
    ]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

reflection_prompt_template = """
You are analyzing conversations about research papers to create memories that will help guide future interactions. Your task is to extract key elements that would be most helpful when encountering similar academic discussions in the future.

Review the conversation and create a memory reflection following these rules:

1. For any field where you don't have enough information or the field isn't relevant, use "N/A"
2. Be extremely concise - each string should be one clear, actionable sentence
3. Focus only on information that would be useful for handling similar future conversations
4. Context_tags should be specific enough to match similar situations but general enough to be reusable

Output valid JSON in exactly this format:
{{
    "context_tags": [              // 2-4 keywords that would help identify similar future conversations
        string,                    // Use field-specific terms like "deep_learning", "methodology_question", "results_interpretation"
        ...
    ],
    "conversation_summary": string, // One sentence describing what the conversation accomplished
    "what_worked": string,         // Most effective approach or strategy used in this conversation
    "what_to_avoid": string        // Most important pitfall or ineffective approach to avoid
}}

Examples:
- Good context_tags: ["transformer_architecture", "attention_mechanism", "methodology_comparison"]
- Bad context_tags: ["machine_learning", "paper_discussion", "questions"]

- Good conversation_summary: "Explained how the attention mechanism in the BERT paper differs from traditional transformer architectures"
- Bad conversation_summary: "Discussed a machine learning paper"

- Good what_worked: "Using analogies from matrix multiplication to explain attention score calculations"
- Bad what_worked: "Explained the technical concepts well"

- Good what_to_avoid: "Diving into mathematical formulas before establishing user's familiarity with linear algebra fundamentals"
- Bad what_to_avoid: "Used complicated language"

Additional examples for different research scenarios:

Context tags examples:
- ["experimental_design", "control_groups", "methodology_critique"]
- ["statistical_significance", "p_value_interpretation", "sample_size"]
- ["research_limitations", "future_work", "methodology_gaps"]

Conversation summary examples:
- "Clarified why the paper's cross-validation approach was more robust than traditional hold-out methods"
- "Helped identify potential confounding variables in the study's experimental design"

What worked examples:
- "Breaking down complex statistical concepts using visual analogies and real-world examples"
- "Connecting the paper's methodology to similar approaches in related seminal papers"

What to avoid examples:
- "Assuming familiarity with domain-specific jargon without first checking understanding"
- "Over-focusing on mathematical proofs when the user needed intuitive understanding"

Do not include any text outside the JSON object in your response.

Here is the prior conversation:

{conversation}
"""

reflection_prompt = ChatPromptTemplate.from_template(reflection_prompt_template)

reflect = reflection_prompt | llm | JsonOutputParser()

In [ ]:
!pip install weaviate-client
import weaviate
import os # Ensure os is imported for API key

# --- Original problematic line: ---
# vdb_client = weaviate.connect_to_local()
# print("Connected to Weviate: ", vdb_client.is_ready())

# --- Recommended fix for Colab: Connect to Weaviate Cloud (WCS) ---
# You need a Weaviate Cloud account and a cluster URL/API key.
# Sign up at: https://console.weaviate.cloud/
# Replace 'YOUR_WEAVIATE_CLUSTER_URL' and 'YOUR_WEAVIATE_API_KEY' with your actual credentials.

# To avoid exposing sensitive keys, consider using Colab secrets or environment variables.
# For example, create a Colab Secret named 'WEAVIATE_CLUSTER_URL' and 'WEAVIATE_API_KEY'.
WEAVIATE_CLUSTER_URL = os.getenv("WEAVIATE_CLUSTER_URL", "YOUR_WEAVIATE_CLUSTER_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY", "YOUR_WEAVIATE_API_KEY")

if WEAVIATE_CLUSTER_URL == "YOUR_WEAVIATE_CLUSTER_URL" or WEAVIATE_API_KEY == "YOUR_WEAVIATE_API_KEY":
    print("WARNING: Weaviate credentials are not set. Please update WEAVIATE_CLUSTER_URL and WEAVIATE_API_KEY.")
    vdb_client = None # Set to None to allow subsequent cells to run, but they will fail if they use vdb_client
else:
    try:
        vdb_client = weaviate.connect_to_wcs(
            cluster_url=WEAVIATE_CLUSTER_URL,
            auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY)
        )
        print("Connected to Weaviate Cloud: ", vdb_client.is_ready())
    except Exception as e:
        print(f"Error connecting to Weaviate Cloud: {e}")
        vdb_client = None # Set to None on connection failure

In [ ]:
from weaviate.classes.config import Property, DataType, Configure, Tokenization

vdb_client.collections.create(
    name="episodic_memory",
    description="Collection containing historical chat interactions and takeaways.",
    vectorizer_config=[
        Configure.NamedVectors.text2vec_ollama(
            name="title_vector",
            source_properties=["title"],
            api_endpoint="http://host.docker.internal:11434",  # If using Docker, use this to contact your local Ollama instance
            model="nomic-embed-text",
        )
    ],
    properties=[
        Property(name="conversation", data_type=DataType.TEXT),
        Property(name="context_tags", data_type=DataType.TEXT_ARRAY),
        Property(name="conversation_summary", data_type=DataType.TEXT),
        Property(name="what_worked", data_type=DataType.TEXT),
        Property(name="what_to_avoid", data_type=DataType.TEXT),

    ]
)

In [ ]:
def format_conversation(messages):

    # Create an empty list placeholder
    conversation = []

    # Start from index 1 to skip the first system message
    for message in messages[1:]:
        conversation.append(f"{message.type.upper()}: {message.content}")

    # Join with newlines
    return "\n".join(conversation)

In [ ]:
def add_episodic_memory(messages, vdb_client):

    # Format Messages
    conversation = format_conversation(messages)

    # Create Reflection
    reflection = reflect.invoke({"conversation": conversation})

    # Load Database Collection
    episodic_memory = vdb_client.collections.get("episodic_memory")

    # Insert Entry Into Collection
    episodic_memory.data.insert({
        "conversation": conversation,
        "context_tags": reflection['context_tags'],
        "conversation_summary": reflection['conversation_summary'],
        "what_worked": reflection['what_worked'],
        "what_to_avoid": reflection['what_to_avoid'],
    })

# add_episodic_memory(messages, vdb_client)

In [ ]:
# def stream_llm(messages):
#     full_content = ""

#     print("\n🟨 ASSISTANT (streaming): ", end="", flush=True)

#     for chunk in llm_with_tools.stream(messages):
#         if chunk.content:
#             print(chunk.content, end="", flush=True)
#             full_content += chunk.content

#     print("\n")  # newline after stream
#     return AIMessage(content=full_content)


In [ ]:
# from langchain_core.messages import SystemMessage, AIMessage

# def supervisor_node(state: State):
#     print("🧠 SUPERVISOR CALLED")

#     messages = state["messages"]
#     if not messages:
#         messages = [SystemMessage(content=SUPERVISOR_PROMPT)]

#     # news = state.get("news")
#     # if news["status"] == "NO_DATA":
#     # # HARD STOP — no retry
#     #     return {
#     #     "final_answer": (
#     #         "There is currently no reliable or confirmed news available "
#     #         "on this topic. Based on historical patterns and general "
#     #         "market behavior, here is a cautious, non-real-time perspective..."
#     #     )
#     # }


#     # # 1️⃣ STOP CONDITION (THIS IS THE KEY)
#     # if state.get("news_done"):
#     #     print("🛑 News already fetched. Ending flow.")
#     #     return {
#     #         "messages": messages
#     #     }

#     # 2️⃣ Ensure system prompt
#     if not messages or not isinstance(messages[0], SystemMessage):
#         messages = [SystemMessage(content=SUPERVISOR_PROMPT)] + messages

#     # 3️⃣ Call LLM (routing / tool decision)
#     response = llm_with_tools.invoke(messages)

#     # 5️⃣ Otherwise allow tool execution
#     return {
#         "messages": messages + [response]
#     }


In [ ]:
from langchain_core.messages import SystemMessage, AIMessage
def supervisor_node(state: State):
    print("🧠 SUPERVISOR CALLED")

    messages = state["messages"]
    # structured_memory = state.get("structured_memory", {})
    episodic_context = state.get("episodic_context", "N/A")

    # Inject memory-aware system prompt
    memory_system_prompt = SystemMessage(
        content=SUPERVISOR_PROMPT.format(
            # structured_memory=structured_memory,
            episodic_context=episodic_context
        )
    )

    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [memory_system_prompt] + messages
    else:
        messages[0] = memory_system_prompt

    response = llm_with_tools.invoke(messages)

    return {
        **state,
        "messages": messages + [response]
    }

In [ ]:
def memory_update_node(state: State):
    messages = state["messages"]

    # ---------- Episodic storage ----------
    add_episodic_memory(messages, vdb_client)

    # ---------- Structured extraction ----------
    # conversation_text = format_conversation(messages)
    # extracted = extract_structured_data(conversation_text)

    # memory_manager.update_structured_memory(extracted)

    return state

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


graph = StateGraph(State)

graph.add_node("respond", supervisor_node)
graph.add_node("tools",ToolNode(tools))

graph.set_entry_point("respond")

graph.add_conditional_edges(
    "respond",
    tools_condition
)
graph.add_edge("tools","respond")
graph.add_edge("respond", END)

app = graph.compile()

from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

ValueError: Channel 'messages' already exists with a different type

In [ ]:
# state = {"messages": [{"role": "user", "content": "Do we have to invest in tesla or not based on current conditions?"}]}
# result = app.invoke(state)
# print("Response:", result["messages"][-1].content)

In [ ]:
# from langchain_core.messages import HumanMessage

# result = app.invoke({
#     "messages": [
#         HumanMessage(content="What happens to markets during quantitative tightening?")
#     ]
# })
# print("Response:", result["messages"][-1].content)